# Emerging Tech Lab - Kinetic Opentrons Protocol

## Kinetic protocol setup

In [ ]:
from kinetic_opentrons_helpers import (
    log_step,
    build_target_wells,
    build_timepoint_map,
    validate_volume_for_p300,
    dispense_to_wells_with_tip_changes,
    set_robot_speeds,
    log_absolute_time_tick,
    log_reverse_dialdehyde_order,
    summarise_middle_replicate_finish_times,
)

import opentrons.execute
protocol = opentrons.execute.get_protocol_api("2.19") # Restarting the kernel again.

# Loading labware
plate_48 = protocol.load_labware(
    "greenaway_48_wellplate_3750ul",
    location = 2
)
plate_8 = protocol.load_labware(
    "greenaway_8_wellplate_20000ul",
    location = 3
)
pip_rack = protocol.load_labware(
    "opentrons_96_tiprack_300ul",
    location = 1
)

# Load Pipettes
pip_300 = protocol.load_instrument(
    "p300_single_gen2",
    "left", # Change "right" or "left" depending on which arm the 300 uL pipette is
    tip_racks = [pip_rack]
)

# -----------------------------
# User setup: liquid handling
# -----------------------------

# Optimised handling parameters for volatile organic solvent mixtures
ASPIRATE_RATE = 70          # uL/s
STANDARD_DISPENSE_RATE = 70 # uL/s
SLOW_DISPENSE_RATE = 10     # uL/s, used for dropwise dialdehyde addition
AIR_GAP_VOLUME = 15         # uL
MAX_DISPENSE = 200          # uL; two thirds of the P300 volume
PRE_WET_CYCLES = 3
PRE_WET_VOLUME = 180        # uL
TIP_CHANGE_INTERVAL = 3     # compromise: discard tip after every 3 target-well dispenses
TRANSFER_MODE = "fast"      # "fast" = one tip per reagent/solvent source; "accurate" = change tips regularly

pip_300.flow_rate.aspirate = ASPIRATE_RATE
pip_300.flow_rate.dispense = STANDARD_DISPENSE_RATE

# Speed up robot movement while keeping liquid-handling flow rates controlled.
# Rim-touching during pre-wetting is slowed separately inside the helper function.
set_robot_speeds(
    protocol=protocol,
    pipette=pip_300,
    pipette_default_speed=400,
    max_head_speed=400,
)

# -----------------------------
# User setup: source locations
# -----------------------------

# 8-well source plate layout, can be changed if needed:
# A1: diamine in CHCl3
# A2: diamine in MeOH
# A3: diamine in 1:1 CHCl3/MeOH
# B1: dialdehyde in CHCl3
# B2: dialdehyde in MeOH
# B3: dialdehyde in 1:1 CHCl3/MeOH

SOLVENT_CONDITIONS = {
    "chloroform": {
        "diamine_source": plate_8["A1"],
        "dialdehyde_source": plate_8["B1"],
        "rows": ["A", "B", "C"],
    },
    "methanol": {
        "diamine_source": plate_8["A2"],
        "dialdehyde_source": plate_8["B2"],
        "rows": ["D", "E", "F"],
    },
    # Use this condition in place of one of the two above if needed.
    # The 48-well plate can fit two solvent conditions per run:
    # 8 time points x 3 repeats x 2 conditions = 48 wells.
    "chloroform_methanol_1to1": {
        "diamine_source": plate_8["A3"],
        "dialdehyde_source": plate_8["B3"],
        "rows": ["D", "E", "F"],
    },
}

# Select exactly two solvent conditions for this run.
# Example layout below:
#   chloroform: A1:C6
CONDITIONS_TO_RUN = ["chloroform"]

if len(CONDITIONS_TO_RUN) == 0:
    raise ValueError("Select at least one solvent condition to run.")
# Time-point labels for records only.
# The robot does not wait for these time intervals; instead, the dialdehyde addition order
# is arranged so that later sampling time points are started first and earlier time points last.
# This makes manual sampling/quenching easier: the 0 min samples are started closest to work-up.
TIMEPOINTS_MIN = [20, 40, 60, 80, 100, 120]

if not 1 <= len(TIMEPOINTS_MIN) <= 8:
    raise ValueError("TIMEPOINTS_MIN must contain between 1 and 8 time points to fit columns 1-8 of the 48-well plate.")

# -----------------------------
# User setup: dispense volumes
# -----------------------------

# Enter the calculated volume for each stock solution per reaction vial.
# These volumes are applied to every replicate/time-point well in this run.
volume_of_diamine = 1000      # uL per well
volume_of_dialdehyde = 200   # uL per well

# -----------------------------
# Build and validate the plate map
# -----------------------------

validate_volume_for_p300(volume_of_diamine, "Diamine", protocol=protocol)
validate_volume_for_p300(volume_of_dialdehyde, "Dialdehyde", protocol=protocol)

all_target_wells = []

for condition_name in CONDITIONS_TO_RUN:
    if condition_name not in SOLVENT_CONDITIONS:
        raise ValueError(f"Unknown solvent condition: {condition_name}")

    condition = SOLVENT_CONDITIONS[condition_name]
    condition["target_wells"] = build_target_wells(
        condition["rows"],
        n_columns=len(TIMEPOINTS_MIN),
    )
    condition["timepoint_map"] = build_timepoint_map(
        condition["rows"],
        TIMEPOINTS_MIN,
    )

    wells = condition["target_wells"]
    all_target_wells.extend(wells)

    log_step(protocol, f"Condition: {condition_name}")
    for entry in condition["timepoint_map"]:
        log_step(protocol,
            f"  {entry['time_min']} min, column {entry['column']}: {', '.join(entry['wells'])}"
        )

if len(set(all_target_wells)) != len(all_target_wells):
    raise ValueError("Duplicate target wells detected. Check the selected solvent-condition layouts.")

## Automated kinetic reaction setup

In [ ]:
# -----------------------------
# Automated kinetic reaction setup
# Correct addition order:
#   1. diamine solution
#   2. dialdehyde solution, slow/dropwise
# -----------------------------

# Store dispense records for later inspection.
diamine_dispense_records = {}
dialdehyde_dispense_records = {}
kinetic_start_time_summaries = {}

# Add diamine solution to all wells first.
# This does not start the imine reaction until the dialdehyde is later added.
for condition_name in CONDITIONS_TO_RUN:
    condition = SOLVENT_CONDITIONS[condition_name]
    diamine_dispense_records[condition_name] = dispense_to_wells_with_tip_changes(
        pipette=pip_300,
        protocol=protocol,
        plate=plate_48,
        source_well=condition["diamine_source"],
        target_well_names=condition["target_wells"],
        total_volume=volume_of_diamine,
        dispense_rate=STANDARD_DISPENSE_RATE,
        reagent_name=f"diamine stock ({condition_name})",
        tip_change_interval=TIP_CHANGE_INTERVAL,
        transfer_mode=TRANSFER_MODE,
        max_dispense=MAX_DISPENSE,
        air_gap_volume=AIR_GAP_VOLUME,
        pre_wet_cycles=PRE_WET_CYCLES,
        pre_wet_volume=PRE_WET_VOLUME,
        log_each_dispense_time=False,
        log_absolute_time=False,
    )

# Refined kinetic timing logic:
# The reaction starts when dialdehyde is added.
# For easier manual sampling/quenching, start the longest nominal time points first
# and the shortest/0 min time points last.
for condition_name in CONDITIONS_TO_RUN:
    condition = SOLVENT_CONDITIONS[condition_name]

    dialdehyde_ordered_wells = log_reverse_dialdehyde_order(
        condition["timepoint_map"],
        protocol=protocol,
    )

    log_absolute_time_tick(protocol, f"Starting dialdehyde addition for {condition_name}.")

    dialdehyde_dispense_records[condition_name] = dispense_to_wells_with_tip_changes(
        pipette=pip_300,
        protocol=protocol,
        plate=plate_48,
        source_well=condition["dialdehyde_source"],
        target_well_names=dialdehyde_ordered_wells,
        total_volume=volume_of_dialdehyde,
        dispense_rate=SLOW_DISPENSE_RATE,
        reagent_name=f"dialdehyde stock ({condition_name})",
        tip_change_interval=TIP_CHANGE_INTERVAL,
        transfer_mode=TRANSFER_MODE,
        max_dispense=MAX_DISPENSE,
        air_gap_volume=AIR_GAP_VOLUME,
        pre_wet_cycles=PRE_WET_CYCLES,
        pre_wet_volume=PRE_WET_VOLUME,
        log_each_dispense_time=True,
        log_absolute_time=True,
    )

    kinetic_start_time_summaries[condition_name] = summarise_middle_replicate_finish_times(
        dispense_records=dialdehyde_dispense_records[condition_name],
        timepoint_map=condition["timepoint_map"],
        protocol=protocol,
    )

    log_absolute_time_tick(protocol, f"Finished dialdehyde addition for {condition_name}.")

# Reset dispense rate and home the robot.
pip_300.flow_rate.dispense = STANDARD_DISPENSE_RATE

log_step(protocol, "Kinetic start-time summary by condition:")
for condition_name, summary in kinetic_start_time_summaries.items():
    log_step(protocol, f"Condition: {condition_name}")
    for record in summary:
        log_step(
            protocol,
            f"  nominal {record['time_min']} min | representative well {record['representative_well']} | "
            f"dispense order {record['dispense_order_index']} | start timestamp {record['finish_timestamp']}"
        )

protocol.home()